In [1]:
import os
import copy
import pickle
import sympy
import functools
import itertools
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from error_injection import MissingValueError, SamplingError, Injector
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import mutual_info_score, auc, roc_curve, roc_auc_score, f1_score
from scipy.optimize import minimize as scipy_min
from scipy.spatial import ConvexHull
from scipy.optimize import minimize, Bounds, linprog
from sympy import Symbol as sb
from sympy import lambdify
from tqdm.notebook import trange,tqdm
from IPython.display import display,clear_output
from random import choice
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import _tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score
from itertools import combinations, product
import heapq
class style():
    RED = '\033[31m'
    GREEN = '\033[32m'
    BLUE = '\033[34m'
    RESET = '\033[0m'

np.random.seed(1)

# ignore all the warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
# first impute the data and make it hypothetically clean
def load_mpg_cleaned():
    # fetch dataset
    auto_mpg = pd.read_csv('auto-mpg.csv').drop('car name', axis=1).replace('?', np.nan)
    
    features = ['cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model year', 'origin']
    X = auto_mpg[features].astype(float)
    y = auto_mpg['mpg']
    
    # assumed gt imputation
    imputer = KNNImputer(n_neighbors=10)
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    X_train = copy.deepcopy(X_train).reset_index(drop=True)
    X_test = copy.deepcopy(X_test).reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = load_mpg_cleaned()
print(X_train.shape)
X_train.head()

(318, 7)


,cylinders,displacement,horsepower,weight,acceleration,model year,origin
0,8.0,350.0,125.0,3900.0,17.4,79.0,1.0
1,8.0,455.0,225.0,3086.0,10.0,70.0,1.0
2,4.0,91.0,68.0,2025.0,18.2,82.0,3.0
3,4.0,122.0,86.0,2226.0,16.5,72.0,1.0
4,4.0,97.0,67.0,2065.0,17.8,81.0,3.0


In [3]:
#Useful functions
symbol_id = -1
def create_symbol(suffix=''):
    global symbol_id
    symbol_id += 1
    name = f'e{symbol_id}_{suffix}' if suffix else f'e{symbol_id}'
    return sympy.Symbol(name=name)


scaler_symbols = set([sb(f'k{i}') for i in range(X_train.shape[1]+1)])
linearization_dict = dict()
reverse_linearization_dict = dict()

def inject_sensitive_ranges(X, y, uncertain_attr, uncertain_num, boundary_indices, uncertain_radius_pct=None, 
                  uncertain_radius=None, seed=42):
    global symbol_id
    symbol_id = -1
    
    X_extended = np.append(np.ones((len(X), 1)), X, axis=1)
    ss = StandardScaler()
    X_extended[:, 1:] = ss.fit_transform(X_extended[:, 1:])
    X_extended_symb = sympy.Matrix(X_extended)
    
    if not(uncertain_attr=='y'):
        uncertain_attr_idx = X.columns.to_list().index(uncertain_attr) + 1
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(np.max(X_extended[:, uncertain_attr_idx])-\
                                                     np.min(X_extended[:, uncertain_attr_idx]))
    else:
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(y_train.max()-y_train.min())[0]
    
    np.random.seed(seed)
    uncertain_indices = boundary_indices[:uncertain_num]
    y_symb = sympy.Matrix(y)
    symbols_in_data = set()
    #print(uncertain_indices)
    for uncertain_idx in uncertain_indices:
        new_symb = create_symbol()
        symbols_in_data.add(new_symb)
        if uncertain_attr=='y':
            y_symb[uncertain_idx] = y_symb[uncertain_idx] + uncertain_radius*new_symb
        else:
            X_extended_symb[uncertain_idx, uncertain_attr_idx] = X_extended_symb[uncertain_idx, uncertain_attr_idx] + uncertain_radius*new_symb
    return X_extended_symb, y_symb, symbols_in_data, ss

# if interval=True, use interval arithmetic, otherwise use zonotopes
def compute_robustness_ratio_sensitive_label_error(X_train, y_train, X_test, y_test, robustness_radius,
                                         uncertain_num, boundary_indices, uncertain_radius=None, 
                                         lr=0.1, seed=42, interval=True):
    X, y, symbols_in_data, ss = inject_sensitive_ranges(X=X_train, y=y_train, uncertain_attr='y', 
                                              uncertain_num=uncertain_num, boundary_indices=boundary_indices, 
                                              uncertain_radius=uncertain_radius, 
                                              uncertain_radius_pct=None, seed=seed)
    
    assert len(X.free_symbols)==0
    # closed-form
    param = (X.T*X).inv()*X.T*y
    
    if interval:
        # make param intervals
        for d in range(len(param)):
            expr = param[d]
            if not(expr.free_symbols):
                continue
            else:
                constant_part = 0
                interval_radius = 0
                for arg in expr.args:
                    if arg.free_symbols:
                        interval_radius += abs(arg.args[0])
                    else:
                        assert constant_part == 0
                        constant_part = arg
                param[d] = constant_part + create_symbol()*interval_radius
    
    test_preds = sympy.Matrix(np.append(np.ones((len(X_test), 1)), ss.transform(X_test), axis=1))*param
    robustness_ls = []
    for pred in test_preds:
        pred_range_radius = 0
        for arg in pred.args:
            if arg.free_symbols:
                pred_range_radius += abs(arg.args[0])
        if pred_range_radius <= robustness_radius:
            robustness_ls.append(1)
        else:
            robustness_ls.append(0)
    
#     print(param)
    return np.mean(robustness_ls)

def inject_ranges(X, y, uncertain_attr, uncertain_num, uncertain_radius_pct=None, uncertain_radius=None, seed=42):
    global symbol_id
    symbol_id = -1
    
    X_extended = np.append(np.ones((len(X), 1)), X, axis=1)
    ss = StandardScaler()
    X_extended[:, 1:] = ss.fit_transform(X_extended[:, 1:])
    X_extended_symb = sympy.Matrix(X_extended)
    
    if not(uncertain_attr=='y'):
        uncertain_attr_idx = X.columns.to_list().index(uncertain_attr) + 1
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(np.max(X_extended[:, uncertain_attr_idx])-\
                                                     np.min(X_extended[:, uncertain_attr_idx]))
    else:
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(y_train.max()-y_train.min())[0]
    
    np.random.seed(seed)
    uncertain_indices = np.random.choice(range(len(y)), uncertain_num, replace=False)
    y_symb = sympy.Matrix(y)
    symbols_in_data = set()
    for uncertain_idx in uncertain_indices:
        new_symb = create_symbol()
        symbols_in_data.add(new_symb)
        if uncertain_attr=='y':
            y_symb[uncertain_idx] = y_symb[uncertain_idx] + uncertain_radius*new_symb
        else:
            X_extended_symb[uncertain_idx, uncertain_attr_idx] = X_extended_symb[uncertain_idx, uncertain_attr_idx] + uncertain_radius*new_symb
    return X_extended_symb, y_symb, symbols_in_data, ss

# if interval=True, use interval arithmetic, otherwise use zonotopes
def compute_robustness_ratio_label_error(X_train, y_train, X_test, y_test, robustness_radius,
                                         uncertain_num, uncertain_radius=None, 
                                         lr=0.1, seed=42, interval=True):
    X, y, symbols_in_data, ss = inject_ranges(X=X_train, y=y_train, uncertain_attr='y', 
                                              uncertain_num=uncertain_num, uncertain_radius=uncertain_radius, 
                                              uncertain_radius_pct=None, seed=seed)
    
    assert len(X.free_symbols)==0
    # closed-form
    param = (X.T*X).inv()*X.T*y
    
    if interval:
        # make param intervals
        for d in range(len(param)):
            expr = param[d]
            if not(expr.free_symbols):
                continue
            else:
                constant_part = 0
                interval_radius = 0
                for arg in expr.args:
                    if arg.free_symbols:
                        interval_radius += abs(arg.args[0])
                    else:
                        assert constant_part == 0
                        constant_part = arg
                param[d] = constant_part + create_symbol()*interval_radius
    
    test_preds = sympy.Matrix(np.append(np.ones((len(X_test), 1)), ss.transform(X_test), axis=1))*param
    robustness_ls = []
    for pred in test_preds:
        pred_range_radius = 0
        for arg in pred.args:
            if arg.free_symbols:
                pred_range_radius += abs(arg.args[0])
        if pred_range_radius <= robustness_radius:
            robustness_ls.append(1)
        else:
            robustness_ls.append(0)
    
#     print(param)
    return np.mean(robustness_ls)

In [4]:
X_train.columns

Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
       'model year', 'origin'],
      dtype='object')

In [5]:
X_train_transformed = X_train.copy()

columns_to_bin = ['displacement', 'horsepower', 'weight', 'acceleration']
n_bins = {'displacement': int(np.sqrt(72)), 'horsepower': int(np.sqrt(87)), 'weight': int(np.sqrt(288)), 'acceleration': int(np.sqrt(86))}
column_bins = {}

for col in columns_to_bin:
    # Apply binning (equal-width bins in this example)
    X_train_transformed[col], bins = pd.cut(
        X_train[col], 
        bins=n_bins[col], 
        labels=False, 
        retbins=True
    )

    column_bins[col] = bins

In [6]:
def compute_optimized_candidates(X, tau, n_features_to_combine=2):

    candidates = []
    n_samples, n_features = X.shape
    
    # Generate combinations of features to combine
    feature_combinations = list(combinations(range(n_features), n_features_to_combine))
    
    for feature_combo in tqdm(feature_combinations, desc="Processing feature combinations"):
        # Get unique values for each feature in the combination
        values_list = [np.unique(X.iloc[:, idx]) for idx in feature_combo]
        
        # Iterate over all possible value-condition combinations
        for value_combo in tqdm(
            product(*values_list),
            desc=f"Processing combination {feature_combo}",
            leave=False
        ):
            for conditions in product(["<", "=", ">"], repeat=n_features_to_combine):
                # Skip conflicting combinations
                if has_conflicts(value_combo, conditions):
                    continue

                # Generate pattern
                pattern = np.ones(n_samples, dtype=bool)  # Start with all True
                for idx, (feat_idx, val, cond) in enumerate(zip(feature_combo, value_combo, conditions)):
                    if cond == "<":
                        pattern &= X.iloc[:, feat_idx] < val
                    elif cond == "=":
                        pattern &= X.iloc[:, feat_idx] == val
                    elif cond == ">":
                        pattern &= X.iloc[:, feat_idx] > val
                
                # Check support
                support = np.sum(pattern)
                if support >= tau * n_samples and support != n_samples:
                    candidates.append((feature_combo, value_combo, conditions))
    
    return candidates


def has_conflicts(value_combo, conditions):
    for i in range(len(value_combo)):
        for j in range(i + 1, len(value_combo)):
            # Detect conflicts like "< x AND > x" for the same value
            if conditions[i] == ">" and conditions[j] == "<" and value_combo[i] <= value_combo[j]:
                return True
            if conditions[i] == "<" and conditions[j] == ">" and value_combo[i] >= value_combo[j]:
                return True
            # Detect conflicts like "= x AND < x" or "= x AND > x"
            if conditions[i] == "=" and (conditions[j] == "<" and value_combo[i] >= value_combo[j]):
                return True
            if conditions[i] == "=" and (conditions[j] == ">" and value_combo[i] <= value_combo[j]):
                return True
            # Symmetry of conditions
            if conditions[j] == "=" and (conditions[i] == "<" and value_combo[j] >= value_combo[i]):
                return True
            if conditions[j] == "=" and (conditions[i] == ">" and value_combo[j] <= value_combo[i]):
                return True
    return False

In [7]:
candidates = compute_optimized_candidates(X_train_transformed, 0.1, n_features_to_combine=3)
len(candidates)

Processing feature combinations:   0%|          | 0/35 [00:00<?, ?it/s]

Processing combination (0, 1, 2): 0it [00:00, ?it/s]

Processing combination (0, 1, 3): 0it [00:00, ?it/s]

Processing combination (0, 1, 4): 0it [00:00, ?it/s]

Processing combination (0, 1, 5): 0it [00:00, ?it/s]

...

Processing combination (3, 4, 5): 0it [00:00, ?it/s]

Processing combination (3, 4, 6): 0it [00:00, ?it/s]

Processing combination (3, 5, 6): 0it [00:00, ?it/s]

Processing combination (4, 5, 6): 0it [00:00, ?it/s]

25304

In [8]:
def filter_dataframe_by_complex_candidate(df, candidate):
    columns, values, conditions = candidate
    mask = pd.Series(True, index=df.index)  # Start with a mask that selects all rows

    for col, val, cond in zip(columns, values, conditions):
        if cond == "<":
            mask &= df.iloc[:, col] < val
        elif cond == "=":
            mask &= df.iloc[:, col] == val
        elif cond == ">":
            mask &= df.iloc[:, col] > val
        else:
            raise ValueError(f"Unsupported condition: {cond}")

    return df[mask].index.tolist()

In [9]:
def top_k_finder(candidates, X_transformed, X_train, y_train, X_test, y_test, k, tau):
    n_samples, n_features = X_train.shape

    # Min-heap to store the top k worst robustness ratios
    top_k_patterns = []
    
    for candidate in tqdm(candidates, desc="Processing Candidates"):
        # Get indices satisfying the candidate pattern
        indices = filter_dataframe_by_complex_candidate(X_transformed, candidate)
        
        # Select boundary indices for robustness calculation
        if len(indices) < int(0.1 * n_samples):  # Skip if not enough data
            continue
        boundary_indices = random.sample(list(indices), int(0.1 * n_samples))
        
        # Compute robustness ratio
        robustness_ratio = compute_robustness_ratio_sensitive_label_error(
            X_train, y_train, X_test, y_test, 
            uncertain_num=int(0.1 * len(y_train)),
            boundary_indices=boundary_indices,
            uncertain_radius=0.25 * (y_train.max() - y_train.min()), 
            robustness_radius=2, 
            interval=False
        )
        
        # Maintain the top k worst robustness ratios (largest values)
        if len(top_k_patterns) < k:
            # Add to heap if less than k elements
            heapq.heappush(top_k_patterns, (robustness_ratio, candidate))
        else:
            # Replace smallest if current ratio is larger
            heapq.heappushpop(top_k_patterns, (robustness_ratio, candidate))
    
    # Sort results in descending order of robustness ratio
    top_k_patterns.sort(reverse=True, key=lambda x: x[0])
    return top_k_patterns

In [ ]:
result = top_k_finder(candidates[0:3000], X_train_transformed, X_train, y_train, X_test, y_test, 10, 0.1)

Processing Candidates:   0%|          | 0/3000 [00:00<?, ?it/s]

In [ ]:
result